# Compare aFRR VWAP vs Average Activation Price

Objective: validate that manually derived hourly VWAP from 15-minute
Regelleistung data is the most robust work-price signal for modeling.

In [ ]:
from pathlib import Path
import os
import requests
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv()

## 1) ENTSO-E CBMP failure reproduction (why not CBMP)

In [ ]:
token = os.getenv("ENTSOE_API_TOKEN") or os.getenv("ENTSOE_API_KEY")
if not token:
    print("ENTSOE token missing (ENTSOE_API_TOKEN/ENTSOE_API_KEY).")
else:
    base_url = "https://web-api.tp.entsoe.eu/api"
    # TenneT SCA, one-month request
    params = {
        "securityToken": token,
        "documentType": "A84",
        "processType": "A67",
        "businessType": "A96",
        "Standard_MarketProduct": "A01",
        "controlArea_Domain": "10YDE-EON------1",
        "periodStart": "202401010000",
        "periodEnd": "202402010000",
    }
    try:
        r = requests.get(base_url, params=params, timeout=90)
        r.raise_for_status()
        print("CBMP request unexpectedly succeeded. Status:", r.status_code)
        print("Response head:")
        print(r.text[:400])
    except Exception as exc:
        print("CBMP request failed as expected:")
        print(type(exc).__name__, str(exc))

**Conclusion:** The CBMP via ENTSO-E was rejected due to persistent API
timeouts and non-standard 4-second resolutions (PT4S) from other German TSOs.

## 2) Load 15-minute Regelleistung price/volume and compute hourly VWAP

In [ ]:
path_15m = Path("data/raw/regelleistung_15min/afrr_price_volume_15min.parquet")
if not path_15m.exists():
    raise FileNotFoundError(f"Missing file: {path_15m}")

raw = pl.read_parquet(path_15m).with_columns(
    pl.col("timestamp_utc").cast(pl.Datetime(time_unit="us", time_zone="UTC"), strict=False)
).sort("timestamp_utc")

required = ["abgerufene_arbeit_pos", "arbeitspreis_pos"]
missing = [c for c in required if c not in raw.columns]
if missing:
    raise KeyError(f"Missing required columns in 15-min parquet: {missing}")

print("Rows:", raw.height)
print("Columns:", raw.columns)

base = raw.select([
    "timestamp_utc",
    pl.col("abgerufene_arbeit_pos").cast(pl.Float64, strict=False),
    pl.col("arbeitspreis_pos").cast(pl.Float64, strict=False),
    (
        pl.col("durchschnittlicher_arbeitspreis_pos").cast(pl.Float64, strict=False)
        if "durchschnittlicher_arbeitspreis_pos" in raw.columns
        else pl.lit(None).cast(pl.Float64).alias("durchschnittlicher_arbeitspreis_pos")
    ),
]).with_columns([
    (pl.col("abgerufene_arbeit_pos") * pl.col("arbeitspreis_pos")).alias("weighted_cost_pos"),
    pl.col("timestamp_utc").dt.truncate("1h").alias("hour_utc"),
])

hourly = (
    base.group_by("hour_utc")
    .agg([
        pl.sum("weighted_cost_pos").alias("sum_weighted_pos"),
        pl.sum("abgerufene_arbeit_pos").alias("sum_volume_pos"),
        pl.mean("arbeitspreis_pos").alias("simple_mean_price_pos"),
        pl.mean("durchschnittlicher_arbeitspreis_pos").alias("mean_avg_price_pos"),
    ])
    .with_columns([
        pl.when(pl.col("sum_volume_pos") != 0)
        .then(pl.col("sum_weighted_pos") / pl.col("sum_volume_pos"))
        .otherwise(pl.col("simple_mean_price_pos"))
        .alias("hourly_vwap_pos"),
    ])
    .sort("hour_utc")
)

comparison_15m = base.join(
    hourly.select(["hour_utc", "hourly_vwap_pos", "simple_mean_price_pos", "mean_avg_price_pos"]),
    on="hour_utc",
    how="left",
).sort("timestamp_utc")

comparison_15m.head()

## 3) Validation windows (6h each)

In [ ]:
def window_table(df: pl.DataFrame, start_utc: str, hours: int = 6) -> pd.DataFrame:
    start_ts = pd.Timestamp(start_utc)
    end_ts = start_ts + pd.Timedelta(hours=hours)
    w = df.filter(
        (pl.col("timestamp_utc") >= pl.lit(start_ts))
        & (pl.col("timestamp_utc") < pl.lit(end_ts))
    ).select([
        "timestamp_utc",
        "abgerufene_arbeit_pos",
        "arbeitspreis_pos",
        "durchschnittlicher_arbeitspreis_pos",
        "hourly_vwap_pos",
        "simple_mean_price_pos",
    ]).sort("timestamp_utc")
    return w.to_pandas().set_index("timestamp_utc")

pre = window_table(comparison_15m, "2021-03-01T10:00:00Z", hours=6)
post = window_table(comparison_15m, "2022-08-01T10:00:00Z", hours=6)

print("Pre-PICASSO: 2021-03-01 10:00-16:00 UTC")
display(pre)

print("Post-PICASSO: 2022-08-01 10:00-16:00 UTC")
display(post)

## 4) Plot: hourly VWAP vs simple hourly mean

In [ ]:
def plot_window(df_win: pd.DataFrame, title: str):
    hourly = (
        df_win
        .assign(hour=lambda x: x.index.floor("1h"))
        .groupby("hour")[["hourly_vwap_pos", "simple_mean_price_pos"]]
        .first()
        .sort_index()
    )
    ax = hourly.plot(marker="o", figsize=(10, 4))
    ax.set_title(title)
    ax.set_ylabel("EUR/MWh")
    ax.set_xlabel("Hour (UTC)")
    ax.grid(True, alpha=0.3)
    plt.show()

plot_window(pre, "Pre-PICASSO (2021-03-01): VWAP vs Simple Mean")
plot_window(post, "Post-PICASSO (2022-08-01): VWAP vs Simple Mean")